In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.functions import col, lit

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/21 21:17:04 WARN Utils: Your hostname, fqworkstation, resolves to a loopback address: 127.0.1.1; using 192.168.31.147 instead (on interface wlp3s0)
26/02/21 21:17:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/21 21:17:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.format("parquet").load("./data/yellow_tripdata_2025-11.parquet")

df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [3]:
df.coalesce(4).write.format("parquet").mode("overwrite").save("./data/yellow_tripdata_2025-11-4-partitions.parquet")

In [4]:
df.createOrReplaceTempView("master_data")

In [5]:
df.count()

4181444

In [6]:
df.filter(
    "tpep_pickup_datetime >= '2025-11-15' AND tpep_pickup_datetime < '2025-11-16'"
).count()

162604

In [7]:
df = df.withColumns({
    'trip_duration': sf.timestamp_diff('MINUTE', col("tpep_pickup_datetime"), col("tpep_dropoff_datetime"))/lit(60)
})

df.sort("trip_duration", ascending=False).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|     trip_duration|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+------------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|              1|    

In [8]:
zone_mapping = spark.read.format("csv").option("header", "true").load("./data/taxi_zone_lookup.csv")

In [9]:
new_df = df.join(zone_mapping, df.PULocationID == zone_mapping.LocationID, "left")

In [10]:
new_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|      trip_duration|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+

In [13]:
new_df.groupBy(col("Zone")).agg(sf.count("*").alias("count")).sort(col("count")).show(truncate=False)

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Green-Wood Cemetery                          |4    |
|Great Kills                                  |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
|Crotona Park                                 |14   |
|Oakwood                                      |14   |
|New Dorp/Midland Beach                       |14   |
|West Brighton                                |14   |
|Willets Point                                |15   |
|Breezy Point/Fort Tilden/Ri